## Causal Comparator: Quickstart Guide

This notebook serves as a practical introduction to the <b>causal comparator</b> package, a tool designed to identify and evaluate structural changes between two causal systems (DAGs).

Whether you are comparing a "Reference" system to a "Treatment" system or monitoring a process over time, this framework helps you distinguish real causal shifts from statistical noise caused by sample size imbalances.

In [58]:
from causal_comparator.data_generation import EdgePerturbationSimulator
import numpy as np
from causal_comparator.discovery import CausalComparator
import lingam 
from lingam.utils import make_dot
from causal_comparator.metrics import evaluate_binary_classification

In [ ]:
# Helper for un-shuffling matrices if your simulator shuffles nodes
def align_matrix(B_est, p):
    idx = np.argsort(p)
    return B_est[idx, :][:, idx]

In [ ]:
# Initialize Simulator
n_nodes = 10
simulator = EdgePerturbationSimulator(n_nodes=n_nodes, noise_type="uniform", rng=np.random.default_rng(42))

# Create graphs: B1_true is the original, B2_true is the modified version as in [1] and [2]
# delta_true marks the exact edges that were removed (edges in B1 not in B2)
B1_true, B2_true, delta_true = simulator.create_graphs(edge_prob=0.2, n_positives=2)

# Generate synthetic data (N1=1000, N2=100) according to [2]
df1, p1 = simulator.simulate_data(B1_true, n_samples=1000)
df2, p2 = simulator.simulate_data(B2_true, n_samples=100)

print(f"Generated data for {n_nodes} nodes.")
print(f"System 1: {df1.shape[0]} samples | System 2: {df2.shape[0]} samples")

Generated data for 10 nodes.
System 1: 1000 samples | System 2: 100 samples


In [ ]:
# Uncomment this line if you want to visualize the graph representation of B1
# make_dot(B1_true)

In [48]:
# Uncomment this line if you want to visualize the graph representation of B2
# make_dot(B2_true)

In [51]:
# Uncomment this line if you want to visualize the graph representation of delta true.
# delta will represent the edges in B1 that are not in B2.
# make_dot(delta_true)

## Causal Comparator

In [20]:
# Initialize Causal Comparator
comparator = CausalComparator(data_i=df1, data_j=df2, model_class=lingam.DirectLiNGAM)


## Execute Naive Estimation

In [ ]:
# Execute Naive Estimation
comparator.estimate_naive()
# Align and Binarize, don't need to align when you aren't simulate data
# with simulate_data
B1_naive = align_matrix(comparator.freq_i, p1)
B2_naive = align_matrix(comparator.freq_j, p2)

# Logic for Naive: Binary indicator of Removal
# 1 if edge is in B1 and NOT in B2, else 0 as in [1](equation 3)
delta_naive = ((B1_naive == 1) & (B2_naive == 0)).astype(int)

### Evaluation Naive Estimation

In [67]:
# evaluation
scores = evaluate_binary_classification(
                                        y_true=delta_true,
                                        y_scores=delta_naive,
                                        optimize=False
                                        )
print(scores)

{'auc_roc': 0.75, 'aupr': 0.51, 'best_f1': 0.6666666666666666, 'best_threshold': 'N/A', 'precision': 1.0, 'recall': 0.5}


## Execute Bootstrapping

In [64]:
# Method 2: Execute Standard Bootstrap
comparator.estimate_bootstrap(n_sampling=100)

# Align
B1_bs = align_matrix(comparator.freq_i, p1)
B2_bs = align_matrix(comparator.freq_j, p2)

# Calculate the Delta Score in the interval [-1,1]
delta_bs = B1_bs - B2_bs

### Evaluation Bootstrapping

In [65]:
# evaluation
scores = evaluate_binary_classification(
                                        y_true=delta_true,
                                        y_scores=delta_bs,
                                        optimize=True
                                        )
print(scores)

{'auc_roc': 1.0, 'aupr': 1.0, 'best_f1': 1.0, 'best_threshold': np.float64(0.27272727272727276), 'precision': 1.0, 'recall': 1.0}


## Execute RSBS

In [60]:
# Method 3: RSBS (Equal sample-size resampling - Recommended)
comparator.estimate_rsbs(n_sampling=10, seed=42)
# Align the resulting frequency matrices back to the canonical node order
# don't need to align when you aren't simulate data with simulate_data
B1_rsbs = align_matrix(comparator.freq_i, p1)
B2_rsbs = align_matrix(comparator.freq_j, p2)

# Calculate the Delta Score in the interval [-1,1]
delta_rsbs = B1_rsbs - B2_rsbs

### Evaluation RSBS

In [62]:
# evaluation
scores = evaluate_binary_classification(
                                        y_true=delta_true,
                                        y_scores=delta_rsbs,
                                        optimize=True
                                        )
print(scores)

{'auc_roc': 0.9387755102040816, 'aupr': 0.5714285714285714, 'best_f1': 0.5, 'best_threshold': np.float64(0.5050505050505051), 'precision': 0.5, 'recall': 0.5}


## References
[1] Ma, Sisi, and Roshan Tourani. "Comparing Causal Bayesian Networks Estimated from Data." Entropy 26.3 (2024): 228.

[2] Shimizu, Shohei, et al. "DirectLiNGAM: A direct method for learning 
a linear non-Gaussian structural equation model." Journal of Machine 
Learning Research-JMLR 12.Apr (2011): 1225-1248.